# Amazon Products Dataset — Exploration & Offline/Online Split

Prepares the `asaniczka/amazon-products-dataset-2023-1-4m-products` Kaggle dataset for benchmarking two ecommerce search architectures:

- **80% — offline ingestion corpus**: indexed into both search architectures ahead of time.
- **20% — online traffic pool**: held out to drive the online/serving side of the experiment.

The split is a single random shuffle with a fixed seed, so it is exactly reproducible across runs and across the two architectures being compared.

## 0. Setup

```bash
pip install -r requirements.txt
```

**Kaggle authentication:** `kagglehub` reads credentials from `~/.kaggle/kaggle.json` or from the `KAGGLE_USERNAME` / `KAGGLE_KEY` environment variables — it does **not** read a single `KAGGLE_API_TOKEN`. If your `.env` only has `KAGGLE_API_TOKEN`, replace it with:

```
KAGGLE_USERNAME=your_kaggle_username
KAGGLE_KEY=your_kaggle_key
```

(from your Kaggle account's API token JSON), or run `kagglehub.login()` interactively.

In [ ]:
import json
import os
import time
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from kagglehub import KaggleDatasetAdapter

DATASET_HANDLE = "asaniczka/amazon-products-dataset-2023-1-4m-products"
PRODUCTS_FILE = "amazon_products.csv"
CATEGORIES_FILE = "amazon_categories.csv"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
OFFLINE_FRACTION = 0.8

load_dotenv(PROJECT_ROOT / ".env")
if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
    print("Warning: KAGGLE_USERNAME / KAGGLE_KEY not set — kagglehub will prompt to authenticate.")

## 1. Quick preview

Load a small sample first to confirm the schema before pulling the full ~1.4M-row file.

In [ ]:
preview_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    DATASET_HANDLE,
    PRODUCTS_FILE,
    pandas_kwargs={"nrows": 1000},
)
preview_df.head()

In [ ]:
preview_df.info()

## 2. Category lookup

In [ ]:
categories_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    DATASET_HANDLE,
    CATEGORIES_FILE,
)
print("categories columns:", categories_df.columns.tolist())
categories_df.head()

## 3. Full load

Loads all rows and attaches the human-readable category name. This is the DataFrame the split is drawn from.

In [ ]:
products_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    DATASET_HANDLE,
    PRODUCTS_FILE,
)

id_col = "id" if "id" in categories_df.columns else categories_df.columns[0]
name_col = "category_name" if "category_name" in categories_df.columns else categories_df.columns[1]

products_df = products_df.merge(
    categories_df[[id_col, name_col]].rename(columns={id_col: "category_id", name_col: "category_name"}),
    how="left",
    on="category_id",
)

print(f"rows={len(products_df):,}  cols={products_df.shape[1]}")
products_df.dtypes

## 4. Data quality checks

In [ ]:
missing_pct = (products_df.isna().mean() * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0].round(2)

In [ ]:
dup_count = products_df.duplicated(subset="asin").sum()
print(f"Duplicate ASINs: {dup_count:,}")

if dup_count:
    products_df = products_df.drop_duplicates(subset="asin", keep="first").reset_index(drop=True)
    print(f"Rows after de-duplication: {len(products_df):,}")

## 5. Exploratory stats & plots

In [ ]:
products_df[["price", "listPrice", "stars", "reviews", "boughtInLastMonth"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

products_df["price"].clip(upper=products_df["price"].quantile(0.99)).hist(bins=50, ax=axes[0])
axes[0].set_title("Price distribution (99th pct clipped)")
axes[0].set_xlabel("price ($)")

products_df["stars"].hist(bins=20, ax=axes[1])
axes[1].set_title("Star rating distribution")
axes[1].set_xlabel("stars")

plt.tight_layout()
plt.show()

In [ ]:
top_categories = products_df["category_name"].value_counts().head(20)
ax = top_categories.plot(kind="barh", figsize=(8, 6))
ax.invert_yaxis()
ax.set_title("Top 20 categories by product count")
ax.set_xlabel("# products")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Best-seller share: {products_df['isBestSeller'].mean():.2%}")
print(f"Products with sales last month > 0: {(products_df['boughtInLastMonth'] > 0).mean():.2%}")

## 6. Randomized 80/20 split

A single fixed-seed permutation of the full (de-duplicated) product set. No stratification is applied — this is a pure random split, so at this dataset size (~1.4M rows) category and price proportions should track the full dataset closely by law of large numbers. The comparison table below is a sanity check, not a correction step.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
shuffled_idx = rng.permutation(len(products_df))
split_point = int(len(products_df) * OFFLINE_FRACTION)

offline_df = products_df.iloc[shuffled_idx[:split_point]].reset_index(drop=True)
online_df = products_df.iloc[shuffled_idx[split_point:]].reset_index(drop=True)

assert set(offline_df["asin"]).isdisjoint(set(online_df["asin"])), "splits overlap"
assert len(offline_df) + len(online_df) == len(products_df)

print(f"offline (ingestion): {len(offline_df):,} rows ({len(offline_df) / len(products_df):.1%})")
print(f"online  (traffic):   {len(online_df):,} rows ({len(online_df) / len(products_df):.1%})")

In [ ]:
comparison = pd.DataFrame({
    "full": products_df["category_name"].value_counts(normalize=True),
    "offline": offline_df["category_name"].value_counts(normalize=True),
    "online": online_df["category_name"].value_counts(normalize=True),
}).fillna(0)
(comparison * 100).round(2).sort_values("full", ascending=False).head(15)

## 7. Persist the split

Written as Parquet (preserves dtypes, compact) alongside a metadata JSON recording the seed and sizes so the split is auditable and reproducible for both architecture runs.

In [ ]:
offline_path = DATA_DIR / "offline_ingestion.parquet"
online_path = DATA_DIR / "online_traffic.parquet"

offline_df.to_parquet(offline_path, index=False)
online_df.to_parquet(online_path, index=False)

metadata = {
    "dataset_handle": DATASET_HANDLE,
    "random_seed": RANDOM_SEED,
    "offline_fraction": OFFLINE_FRACTION,
    "total_rows": len(products_df),
    "offline_rows": len(offline_df),
    "online_rows": len(online_df),
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(DATA_DIR / "split_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

metadata

## 8. Next steps

- `data/offline_ingestion.parquet` — ingest this into both search architectures being compared.
- `data/online_traffic.parquet` — this dataset has no query logs or relevance judgments, only products, so "online traffic" here is a *pool of held-out products*, not ready-made queries. Decide explicitly how queries will be derived from it (e.g. sampled title terms, attribute-based queries) before running the online-side experiment — that choice will materially affect the comparison.
- `data/split_metadata.json` — re-run with the same seed to regenerate an identical split for either architecture.